In [1]:
!pip install sentence-transformers python-docx PyPDF2 pandas
!pip install spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 107.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
df = pd.read_csv("fake_job_postings.csv")  # dosya adı
print(df.columns.tolist())
print(df.shape)
print(df.head(2))

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
(17880, 18)
   job_id                                      title          location  \
0       1                           Marketing Intern  US, NY, New York   
1       2  Customer Service - Cloud Video Production    NZ, , Auckland   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   

                                         description  \
0  Food52, a fast-growing, James Beard Award-winn...   
1  Organised - Focused - Vibrant - Awesome!Do you...   

                                        requirements  \
0  Experience with 

In [3]:
import pandas as pd
import PyPDF2
from docx import Document
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

In [4]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

# Veri setinden skill çıkar
tum_kelimeler = []

for metin in df["requirements"].fillna("").tolist()[:2000]:  # ilk 2000 ilan
    doc = nlp(metin[:500])  # hız için ilk 500 karakter
    for token in doc:
        if (token.pos_ in ["NOUN", "PROPN"]  # isim veya özel isim
            and not token.is_stop             # gereksiz kelime değil
            and not token.is_punct            # noktalama değil
            and len(token.text) > 2):         # çok kısa değil
            tum_kelimeler.append(token.text.lower())

# En sık geçen 100 kelimeyi al
sayac = Counter(tum_kelimeler)
SKILL_LIST = [kelime for kelime, sayi in sayac.most_common(100)]

print(f"Üretilen skill sayısı: {len(SKILL_LIST)}")
print(SKILL_LIST[:30])

Üretilen skill sayısı: 100
['experience', 'skills', 'years', 'knowledge', 'degree', 'ability', 'communication', 'customer', 'sales', 'management', 'team', 'service', 'development', 'amp', 'work', 'web', 'design', 'business', 'environment', 'qualifications', 'computer', 'marketing', 'understanding', 'minimum', 'requirements', 'university', 'software', 'year', 'excel', 'school']


In [5]:
df = pd.read_csv("fake_job_postings.csv")

# Sadece gerçek ilanları kullan (fraudulent=0), description+requirements birleştir
df = df[df["fraudulent"] == 0].copy()
df["full_text"] = df["title"].fillna("") + " " + \
                  df["description"].fillna("") + " " + \
                  df["requirements"].fillna("")
df = df[df["full_text"].str.len() > 100].reset_index(drop=True)

print(f"Kullanılabilir ilan sayısı: {len(df)}")

Kullanılabilir ilan sayısı: 16996


In [6]:
SKILL_LIST = [
    # CV'dekiler
    "python", "c", "c#", "c++", "java", "javascript", "sql",
    "ms sql", "git", "github", "linux", "ubuntu", "vs code",
    "visual studio", "esp32", "freertos", "uart", "i2c", "spi",
    "embedded systems", "sensor", "iot", "cybersecurity",
    "object-oriented programming", "oop", "database",
    # Genel teknik
    "machine learning", "deep learning", "nlp", "tensorflow",
    "pytorch", "docker", "flask", "django", "react", "aws",
    "html", "css", "agile", "scrum", "communication",
    "teamwork", "leadership", "project management",
    "real-time", "networking", "freertos", "rtos",
    "microcontroller", "firmware", "driver"
]

def extract_skills(text):
    text = text.lower()
    return list(set([s for s in SKILL_LIST if s in text]))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

In [7]:
def extract_text_from_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

def extract_text_from_docx(path):
    doc = Document(path)
    return " ".join([p.text for p in doc.paragraphs])

print("CV dosyanızı yükleyin (PDF veya DOCX):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(".pdf"):
    cv_text = extract_text_from_pdf(filename)
elif filename.endswith(".docx"):
    cv_text = extract_text_from_docx(filename)
else:
    cv_text = ""
    print("Hata: Sadece PDF veya DOCX desteklenir!")

cv_skills = extract_skills(cv_text)
print(f"\n✅ CV'de tespit edilen beceriler: {cv_skills}")

CV dosyanızı yükleyin (PDF veya DOCX):


Saving Evla_Karagoz_EvlaKaragoz_CvGomulu.pdf to Evla_Karagoz_EvlaKaragoz_CvGomulu.pdf

✅ CV'de tespit edilen beceriler: ['c', 'freertos', 'github', 'spi', 'ubuntu', 'c#', 'git', 'i2c', 'c++', 'visual studio', 'linux', 'rtos', 'python', 'uart', 'esp32', 'vs code']


In [8]:
print("Model yükleniyor...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# İlk 500 ilan (hız için, tüm dataset çok uzun sürer)
sample_df = df.sample(500, random_state=42).reset_index(drop=True)

print("Embeddingler hesaplanıyor...")
cv_embedding = model.encode([preprocess(cv_text)])
job_embeddings = model.encode(sample_df["full_text"].tolist(), show_progress_bar=True)

scores = cosine_similarity(cv_embedding, job_embeddings)[0]
sample_df["uyum_skoru"] = scores
sample_df = sample_df.sort_values("uyum_skoru", ascending=False).reset_index(drop=True)

print("✅ Tamamlandı!")

Model yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddingler hesaplanıyor...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Tamamlandı!


In [9]:
def analiz_raporu(cv_text, cv_skills, sample_df):
    print("=" * 65)
    print("        📋 SKİLLSYNC — CV ANALİZ RAPORU")
    print("=" * 65)

    best = sample_df.iloc[0]

    # Genel uyum
    print(f"\n🏆 EN YÜKSEK UYUMLU İLAN")
    print(f"   Pozisyon  : {best['title']}")
    print(f"   Lokasyon  : {best['location']}")
    print(f"   Sektör    : {best['industry'] if pd.notna(best['industry']) else 'Belirtilmemiş'}")
    print(f"   Uyum Skoru: %{best['uyum_skoru']*100:.1f}")

    # Skill gap
    best_skills = extract_skills(str(best["full_text"]))
    matched = [s for s in best_skills if s in cv_skills]
    missing = [s for s in best_skills if s not in cv_skills]

    print(f"\n✅ CV'deki Eşleşen Beceriler : {matched if matched else 'Tespit edilemedi'}")
    print(f"❌ CV'de Eksik Beceriler     : {missing if missing else 'Eksik yok'}")

    if missing:
        tahmini = min(best['uyum_skoru'] + len(missing) * 0.04, 1.0)
        print(f"\n💡 Bu eksik beceriler kazanılsaydı tahmini uyum: %{tahmini*100:.1f}")

    # Alan istatistikleri
    print(f"\n📊 ALAN İSTATİSTİKLERİ")
    print(f"   Analiz edilen ilan sayısı     : {len(sample_df)}")
    print(f"   %50 üzeri uyumlu ilan sayısı : {(sample_df['uyum_skoru'] > 0.5).sum()}")
    print(f"   %70 üzeri uyumlu ilan sayısı : {(sample_df['uyum_skoru'] > 0.7).sum()}")
    print(f"   Ortalama uyum skoru           : %{sample_df['uyum_skoru'].mean()*100:.1f}")

    # Sektör dağılımı
    print(f"\n🏭 ALANINIZDAKI EN YAYGIN SEKTÖRLER (Top 5):")
    top_sektorler = sample_df["industry"].value_counts().head(5)
    for sektor, sayi in top_sektorler.items():
        print(f"   {sektor}: {sayi} ilan")

    # Top 5 ilan
    print(f"\n🔝 SİZE EN UYGUN İLK 5 İLAN:")
    for i, row in sample_df.head(5).iterrows():
        print(f"   {i+1}. {row['title']} — %{row['uyum_skoru']*100:.1f} uyum")

    print("\n" + "=" * 65)

analiz_raporu(cv_text, cv_skills, sample_df)

        📋 SKİLLSYNC — CV ANALİZ RAPORU

🏆 EN YÜKSEK UYUMLU İLAN
   Pozisyon  : Advanced Embedded Software Engineer
   Lokasyon  : MX, QUE, Querétaro
   Sektör    : Automotive
   Uyum Skoru: %56.0

✅ CV'deki Eşleşen Beceriler : ['c', 'spi', 'i2c', 'c++', 'uart']
❌ CV'de Eksik Beceriler     : ['communication', 'microcontroller', 'embedded systems', 'teamwork']

💡 Bu eksik beceriler kazanılsaydı tahmini uyum: %72.0

📊 ALAN İSTATİSTİKLERİ
   Analiz edilen ilan sayısı     : 500
   %50 üzeri uyumlu ilan sayısı : 9
   %70 üzeri uyumlu ilan sayısı : 0
   Ortalama uyum skoru           : %28.0

🏭 ALANINIZDAKI EN YAYGIN SEKTÖRLER (Top 5):
   Information Technology and Services: 56 ilan
   Computer Software: 48 ilan
   Internet: 36 ilan
   Marketing and Advertising: 23 ilan
   Education Management: 23 ilan

🔝 SİZE EN UYGUN İLK 5 İLAN:
   1. Advanced Embedded Software Engineer — %56.0 uyum
   2. Software Engineer Intern — %55.0 uyum
   3. Applications Engineer — %54.3 uyum
   4. Programmer — %53.6 

In [10]:
print(cv_text[:1000])

Bilgisayar Mühendisliği 3. sınıf öğrencisi olarak gömülü sistemler ve savunma sanayii teknolojilerine
odaklanmaktayım. Teknofest süreçlerinde aviyonik sistem tasarımı ve Fluxion Lab bünyesinde ESP32 tabanlı
donanım-yazılım entegrasyonu konularında pratik deneyim kazandım. FreeRTOS ile çok görevli (multi-tasking)
uygulama geliştirme, register seviyesinde mikrodenetleyici programlama ve endüstriyel haberleşme protokolleri
üzerine çalışmalar yürütmekteyim. Öğrenmeye açık çalışma anlayışım ve Milli Teknoloji Hamlesi vizyonu
doğrultusunda, kritik sistemlerde yüksek performanslı ve güvenilir gömülü yazılımlar geliştirmeyi hedefliyorum.EVLA KARAGÖZ
İstanbul, Türkiye 
05313553055, ev04.su@gmail.com 
LinkedIn: Evla Karagöz, https://www.linkedin.com/in/evla-karag%C3%B6z-79778b326/BİLGİSAYAR MÜHENDİSLİĞİ
DENEYİM
Fluxion Lab – TÜBİTAK Marmara Teknokent
Gömülü Sistemler Stajyeri (Gönüllü) | Haziran 2025 - Ağustos 2025
RTOS Tabanlı Geliştirme: ESP32-S3 mikrodenetleyicileri üzerinde FreeRTOS kullanar

In [11]:
# Sümeyye'nin CV havuzunu oku
with open("aday_cv_havuzu.txt", "r", encoding="utf-8") as f:
    icerik = f.read()

print(icerik[:1000])  # İlk 1000 karakteri göster

           SİSTEMDE KAYITLI ADAY CV HAVUZU
           Toplam Kayıt Sayısı: 20

           ADAY ÖZGEÇMİŞİ (CV #1)
İSİM SOYİSİM: Ali Yılmaz
E-POSTA: ali.yılmaz@email.com
LOKASYON: GB, , London
HEDEF POZİSYON: Personal Assistant

PROFESYONEL ÖZET:
Citymapper is looking for a Personal Assistant to look after the administrative needs of our CEO and team. We are an early-stage startup, expanding quickly, so a successful candidate will have to be a fast, flexible, organized and motivated individual with a can-do attitude to allow to the CEO and team to focus on what matters the most. Part of the remit will also be to ensure the office runs smoo...

DENEYİM & YETKİNLİKLER:
#NAME?

ŞİRKET KÜLTÜRÜNE UYUM (REFERANS):
We believe cities are complicated. And your mobile device should save you from t


In [13]:
# CV havuzunu parçalara ayır
cv_blocks = icerik.split("=" * 50)
cv_blocks = [b.strip() for b in cv_blocks if "İSİM SOYİSİM" in b]

print(f"Toplam CV sayısı: {len(cv_blocks)}")

# Her CV için analiz yap
for i, cv in enumerate(cv_blocks):
    print(f"\n{'='*60}")
    print(f"TEST CV #{i+1}")
    print('='*60)

    cv_skills = extract_skills(cv)
    cv_embedding = model.encode([preprocess(cv)])
    scores = cosine_similarity(cv_embedding, job_embeddings)[0]
    sample_df["uyum_skoru"] = scores
    sonuclar = sample_df.sort_values("uyum_skoru", ascending=False).reset_index(drop=True)

    best = sonuclar.iloc[0]
    best_skills = extract_skills(str(best["full_text"]))
    missing = [s for s in best_skills if s not in cv_skills]

    print(f"En uyumlu ilan : {best['title']}")
    print(f"Uyum skoru     : %{best['uyum_skoru']*100:.1f}")
    print(f"CV becerileri  : {cv_skills}")
    print(f"Eksik beceriler: {missing}")
    # Sonuçları dosyaya kaydet
with open("test_sonuclari.txt", "w", encoding="utf-8") as f:
    for i, cv in enumerate(cv_blocks):
        cv_skills = extract_skills(cv)
        cv_embedding = model.encode([preprocess(cv)])
        scores = cosine_similarity(cv_embedding, job_embeddings)[0]
        sample_df["uyum_skoru"] = scores
        sonuclar = sample_df.sort_values("uyum_skoru", ascending=False).reset_index(drop=True)
        best = sonuclar.iloc[0]
        best_skills = extract_skills(str(best["full_text"]))
        missing = [s for s in best_skills if s not in cv_skills]

        f.write(f"CV #{i+1}\n")
        f.write(f"En uyumlu ilan: {best['title']}\n")
        f.write(f"Uyum skoru: %{best['uyum_skoru']*100:.1f}\n")
        f.write(f"Eksik beceriler: {missing}\n")
        f.write("="*40 + "\n")

print("Kaydedildi!")

Toplam CV sayısı: 20

TEST CV #1
En uyumlu ilan : Senior Ruby on Rails Developer
Uyum skoru     : %69.1
CV becerileri  : ['c']
Eksik beceriler: ['sql', 'database', 'git']

TEST CV #2
En uyumlu ilan : Big Data - Hadoop Designer
Uyum skoru     : %69.1
CV becerileri  : ['html', 'c', 'communication', 'javascript', 'java', 'c#', 'ms sql', 'css', 'visual studio', 'sql', 'oop']
Eksik beceriler: []

TEST CV #3
En uyumlu ilan : Front-End Engineer
Uyum skoru     : %40.3
CV becerileri  : ['c', 'spi']
Eksik beceriler: ['html', 'javascript', 'java', 'css']

TEST CV #4
En uyumlu ilan : Process Engineer - Environmental laws - TX
Uyum skoru     : %67.3
CV becerileri  : ['c', 'spi', 'communication']
Eksik beceriler: ['aws']

TEST CV #5
En uyumlu ilan : E&IC Engineer
Uyum skoru     : %53.8
CV becerileri  : ['c', 'communication']
Eksik beceriler: []

TEST CV #6
En uyumlu ilan : Sr.Java Developer jobs in Utah
Uyum skoru     : %62.0
CV becerileri  : ['c']
Eksik beceriler: ['javascript', 'java', 'oop']

TES